# Classification Lab

Now that we have discussed the many classification techniques, it is time for us to try them in practice.
We will use the the dataset of toxins and toxin target database ([T3DB](https://www.t3db.ca/)) to classify whether organic compounds are carcinogenic or not. Let's continue working with the same code we had before and try to establish these structure-property relationships

In [1]:
try:
    import google.colab
    IN_COLAB = True
    !git clone https://github.com/dskoda/ml4mat-26s-public.git
    !cd ml4mat-26s-public && pip install . && cd ..
    !pip install xgboost
    !pip install mordred
    ROOT = "https://raw.githubusercontent.com/dskoda/ml4mat-26s-public/refs/heads/main/lectures/06-Kernels"
    STYLE = "colab"
except:
    IN_COLAB = False
    ROOT = "."
    STYLE = "jupyter"

In [2]:
import re
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import cm

# scikit-learn uses a lot of tools that we want
from sklearn import metrics
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold

# RDKit tools
from rdkit.Chem import AllChem as Chem
from rdkit.Chem import Draw

# Mordred to generate molecular descriptors
from mordred import Calculator, descriptors

import ml4mat_ucla as m4m

plt.style.use(STYLE)
m4m.utils.set_dpi(200)

## Loading and cleaning the dataset of toxicity of materials

I have uploaded already the dataset of toxicity of the molecules to make this easier. The dataset contains a lot of information, which you can look at using the `df.head()` analysis:

In [3]:
df = pd.read_json(f"{ROOT}/data/toxins.json.gz")
df.head()

,id,title,common_name,description,cas,pubchem_id,chemical_formula,weight,appearance,melting_point,...,chembl_id,chemspider_id,biodb_id,synthesis_reference,structure_image_caption,synonyms_list,types,cellular_locations,tissues,pathways
0,2,T3D0001,Arsenic,Arsenic(As) is a ubiquitous metalloid found in...,7440-38-2,104734,As,74.921600,Grey metallic solid.,> 615°C,...,,94549,None,None,None,Arsenic ion\r\nArsenic(3+)\r\nArsenic(3+) ion\...,"[{'id': 1, 'type_name': 'Inorganic Compound', ...","[{'id': 1, 'name': 'Cytoplasm', 'created_at': ...",[],[]
1,3,T3D0002,Lead,Lead is a soft and malleable heavy and post-tr...,7439-92-1,73212,Pb,207.976654,"Bluish-white metallic solid, turns grey when e...",327.5°C,...,,65967,None,None,None,Lead (II) cation\r\nLead ion\r\nLead ion (Pb2+...,"[{'id': 1, 'type_name': 'Inorganic Compound', ...","[{'id': 1, 'name': 'Cytoplasm', 'created_at': ...",[],[]
2,4,T3D0003,Mercury,Mercury is a metal that is a liquid at room te...,7439-97-6,26623,Hg,201.970642,Silver metallic liquid.,-38.8°C,...,,24800,None,None,None,Hg(2+)\r\nHg2+\r\nMercuric ion\r\nMercury ion\...,"[{'id': 1, 'type_name': 'Inorganic Compound', ...","[{'id': 1, 'name': 'Cytoplasm', 'created_at': ...",[],[]
3,5,T3D0004,Vinyl chloride,"Vinyl chloride is a man-made organic compound,...",75-01-4,6338,C2H3Cl,61.992330,"Colorless gas, usually stored as a liquid.",-153.7°C,...,None,None,None,,None,Chloroethene\r\nChloroethylene\r\nMonochloroet...,"[{'id': 7, 'type_name': 'Organic Compound', 'c...","[{'id': 1, 'name': 'Cytoplasm', 'created_at': ...",[],[]
4,7,T3D0006,Benzene,"Benzene is a toxic, volatile, flammable liquid...",71-43-2,241,C6H6,78.046950,Colorless liquid.,5.5°C,...,CHEMBL277500,236,None,"Copisarow, Maurice; Long, Cyril N H. The Fried...",None,Annulene\r\nAromatic alkane\r\nBenzeen\r\nBenz...,"[{'id': 7, 'type_name': 'Organic Compound', 'c...","[{'id': 1, 'name': 'Cytoplasm', 'created_at': ...","[{'id': 4, 'name': 'Bone Marrow', 'created_at'...",[]


The data is not fully clean, so we have to deal with it a bit. Specifically, we use the original data to see if something is organic, if it is carcinogenic, and just get floating-point numbers (e.g., remove the deg C)

In [4]:
def is_carcinogenic(txt):
    if txt is None:
        return np.nan
    
    if "carcinogenic to humans" in txt:
        return True
    
    if "possibly carcinogenic" in txt:
        return True
    
    if "probably carcinogenic" in txt:
        return True
    
    if "3, not classifiable" in txt:
        return False
    
    if "No indication" in txt:
        return False
    
    return np.nan

def is_organic(types):
    for t in types:
        if t["type_name"] == "Inorganic Compound":
            return False
            
        if t["type_name"] == "Organic Compound":
            return True

    return False
    
def get_float(txt):
    if type(txt) != str:
        return np.nan
    
    matches = re.findall("[0-9]+", txt)
    if len(matches) > 0:
        return float(matches[0])
    
    return np.nan

In [5]:
df["is_organic"] = df["types"].apply(is_organic)
df["is_carcinogenic"] = df["carcinogenicity"].apply(is_carcinogenic)
df["melting_point"] = df["melting_point"].apply(get_float)
df["boiling_point"] = df["boiling_point"].apply(get_float)

Now, because we have so much information that is not needed, now we limit ourselves to a dataset of organic compounds that contain enough data about them:

In [6]:
relevant_cols = [
    "id",
    "common_name",
    "cas",
    "weight",
    "moldb_smiles",
    "moldb_inchikey",
    "melting_point",
    "boiling_point",
    "is_carcinogenic",
]

org = df.loc[
    (df["is_organic"])
    & (~df["is_carcinogenic"].isna())
    & (~df["moldb_smiles"].isna())
    & (~df["moldb_smiles"].isna())
].reset_index(drop=True)
org = org[relevant_cols]

Let's quickly check the dataset size and how many compounds are carcinogenic there:

In [7]:
frac_positive = org["is_carcinogenic"].sum() / len(org)

print(f"{frac_positive * 100:.0f}% of the {len(org)} compounds are carcinogenic")

23% of the 2720 compounds are carcinogenic


Therefore, the dataset is not balanced.

## Featurizing the dataset

To do any kind of supervised learning, we have to featurize our dataset.
Doing this requires creating molecules from the SMILES, which can be tricky sometimes.
Therefore, we clean up the dataset a bit more by removing the molecules that do not make much sense or cannot be created:

In [8]:
# Creating the mols
mols = [Chem.MolFromSmiles(smi) for smi in org["moldb_smiles"]]

# removing the molecules that are None
indices = list(range(len(org)))
remove = [i for i, mol in enumerate(mols) if mol is None]
indices = list(set(indices) - set(remove))

# updating the list
mols = [mols[i] for i in indices]
org = org.iloc[indices]

[11:08:42] WARNING: not removing hydrogen atom without neighbors
[11:08:42] WARNING: not removing hydrogen atom without neighbors
[11:08:42] WARNING: not removing hydrogen atom without neighbors
[11:08:42] Explicit valence for atom # 7 O, 3, is greater than permitted
[11:08:42] Explicit valence for atom # 6 Al, 6, is greater than permitted
[11:08:42] Explicit valence for atom # 2 Al, 4, is greater than permitted
[11:08:42] Explicit valence for atom # 9 C, 5, is greater than permitted


Finally, we can compute the descriptors using `mordred`:

In [9]:
calc = Calculator(descriptors, ignore_3D=True)
desc = calc.pandas(mols)

 14%|████████████████████████▏                                                                                                                                                  | 384/2716 [00:03<00:09, 239.12it/s]

/opt/homebrew/Caskroom/miniconda/base/envs/ml4mat/lib/python3.10/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


 15%|█████████████████████████▍                                                                                                                                                  | 401/2716 [00:11<02:41, 14.34it/s]

/opt/homebrew/Caskroom/miniconda/base/envs/ml4mat/lib/python3.10/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/opt/homebrew/Caskroom/miniconda/base/envs/ml4mat/lib/python3.10/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/opt/homebrew/Caskroom/miniconda/base/envs/ml4mat/lib/python3.10/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/opt/homebrew/Caskroom/miniconda/base/envs/ml4mat/lib/python3.10/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


 21%|████████████████████████████████████▊                                                                                                                                       | 581/2716 [00:12<00:41, 51.67it/s][11:08:56] WARNING: not removing hydrogen atom without neighbors
[11:08:56] WARNING: not removing hydrogen atom without neighbors
[11:08:56] WARNING: not removing hydrogen atom without neighbors
[11:08:56] WARNING: not removing hydrogen atom without neighbors
[11:08:56] WARNING: not removing hydrogen atom without neighbors
[11:08:56] WARNING: not removing hydrogen atom without neighbors
[11:08:56] WARNING: not removing hydrogen atom without neighbors
[11:08:56] WARNING: not removing hydrogen atom without neighbors
[11:08:56] WARNING: not removing hydrogen atom without neighbors
 25%|███████████████████████████████████████████                                                                                                                                | 684/2716 [00:13<00:18, 10

...

 48%|██████████████████████████████████████████████████████████████████████████████████▋                                                                                        | 1313/2716 [00:18<00:23, 59.43it/s]

/opt/homebrew/Caskroom/miniconda/base/envs/ml4mat/lib/python3.10/site-packages/mordred/_matrix_attributes.py:251: RuntimeWarning: invalid value encountered in scalar power
  s += (eig.vec[i, eig.max] * eig.vec[j, eig.max]) ** -0.5


 95%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌        | 2582/2716 [00:32<00:00, 187.36it/s]

/opt/homebrew/Caskroom/miniconda/base/envs/ml4mat/lib/python3.10/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2716/2716 [00:32<00:00, 82.57it/s]


Again, we have to clean the features - specifically, discarding the structures that failed somehow.
After discarding the features, we still end up with a pretty decent number of features for our dataset:

In [10]:
to_remove = []

for i in range(len(desc)):
    for name, feat in desc.iloc[i].items():
        if not type(feat) in [int, float, np.int64, np.float64]:
            to_remove.append(name)
            
features = desc[[col for col in desc.columns if not col in to_remove]]

In [11]:
features

,nAcid,nBase,nAromAtom,nAromBond,nAtom,nHeavyAtom,nSpiro,nBridgehead,nHetero,nH,...,SRW09,SRW10,TSRW10,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb2
0,0,0,0,0,6,3,0,0,1,3,...,0.000000,4.174387,17.310771,61.992328,10.332055,4,0,6.0,4.0,1.000000
1,0,0,6,6,12,6,0,0,0,6,...,0.000000,7.627057,30.941317,78.046950,6.503913,27,3,24.0,24.0,1.500000
2,0,0,20,24,32,20,0,0,0,12,...,0.000000,10.386808,55.015764,252.093900,7.877934,680,40,120.0,151.0,4.194444
3,0,0,20,22,32,20,0,0,0,12,...,7.260523,10.350382,69.560964,252.093900,7.877934,676,39,120.0,152.0,4.222222
4,0,0,0,0,5,4,0,0,3,1,...,0.000000,6.188264,24.179697,117.914383,23.582877,9,0,12.0,9.0,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2711,0,0,0,0,32,16,0,0,6,16,...,0.000000,9.239025,47.230641,345.938317,10.810572,477,22,70.0,77.0,3.777778
2712,0,0,0,0,41,20,0,0,7,21,...,0.000000,9.409929,52.253174,421.946294,10.291373,939,27,88.0,96.0,4.694444
2713,0,0,0,0,32,14,0,0,4,18,...,0.000000,8.694000,43.624812,278.016261,8.688008,371,14,58.0,59.0,3.277778
2714,0,0,12,12,32,18,0,0,1,14,...,0.000000,9.251386,49.812072,234.104465,7.315765,745,20,84.0,91.0,4.166667


## Training models

Finally, we can train the models.

**Your goal is to train and evaluate the performance of a series of different models**.
Compare them against the metrics we discussed in the previous lecture and in the lecture.

Feel free to ask the instructor if you have any questions!

In [12]:
X = features.values
y = np.array(org["is_carcinogenic"].values).astype(int)

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

models = {
    "logistic": LogisticRegression(),
    "SVC": SVC(),
    "tree": DecisionTreeClassifier(),
    "rf": RandomForestClassifier(),
    "knns": KNeighborsClassifier(),
    "xgboost": XGBClassifier()
}